In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import StratifiedShuffleSplit
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder,StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import root_mean_squared_error
from sklearn.model_selection import cross_val_score

In [2]:
data = pd.read_excel('housing.xlsx')

In [3]:
data['income_cat'] = pd.cut(data['median_income'],
                                bins=[0,1.5,3.0,4.5,6,np.inf],
                                labels=[1,2,3,4,5])

In [4]:
split = StratifiedShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
for train_index, test_index in split.split(data, data['income_cat']):
    strat_train_set = data.loc[train_index].drop("income_cat", axis=1)
    strat_test_set = data.loc[test_index].drop("income_cat", axis=1)

In [5]:
housing = strat_train_set.copy()

In [6]:
housing_labels = housing["median_house_value"].copy()
housing = housing.drop("median_house_value", axis=1)

In [7]:
num_attributes = housing.drop("ocean_proximity", axis=1).columns.tolist()
cat_attributes = ["ocean_proximity"]

In [8]:
num_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy="median")),
    ('std_scaler', StandardScaler()),
])

In [9]:
cat_pipeline = Pipeline([
    ('one_hot', OneHotEncoder(handle_unknown='ignore')),
])

In [10]:
full_pipeline = ColumnTransformer([
    ("num", num_pipeline, num_attributes),
    ("cat", cat_pipeline, cat_attributes),
])

In [11]:
housing_prepared = full_pipeline.fit_transform(housing)

In [12]:
print(housing_prepared)

[[-0.94135046  1.34743822  0.02756357 ...  0.          0.
   0.        ]
 [ 1.17178212 -1.19243966 -1.72201763 ...  0.          0.
   1.        ]
 [ 0.26758118 -0.1259716   1.22045984 ...  0.          0.
   0.        ]
 ...
 [-1.5707942   1.31001828  1.53856552 ...  0.          0.
   0.        ]
 [-1.56080303  1.2492109  -1.1653327  ...  0.          0.
   0.        ]
 [-1.28105026  2.02567448 -0.13148926 ...  0.          0.
   0.        ]]


In [13]:
#Linear regression
lin_reg = LinearRegression()
lin_reg.fit(housing_prepared, housing_labels)
lin_prediction = lin_reg.predict(housing_prepared)
lin_rmse = root_mean_squared_error(housing_labels,lin_prediction)
print("Linear Regression RMSE", lin_rmse)

Linear Regression RMSE 69050.56219504568


In [14]:
#Decision Tree regression
decision_tree_reg = DecisionTreeRegressor()
decision_tree_reg.fit(housing_prepared, housing_labels)
decision_tree_prediction = decision_tree_reg.predict(housing_prepared)
decision_tree_rmse = root_mean_squared_error(housing_labels, decision_tree_prediction)
print("Decision Tree Regression", decision_tree_rmse)

Decision Tree Regression 0.0


In [15]:
#Random forest regression
random_reg = RandomForestRegressor()
random_reg.fit(housing_prepared, housing_labels)
random_prediction = random_reg.predict(housing_prepared)
random_rmse = root_mean_squared_error(housing_labels, random_prediction)
print("Random forest regression", random_rmse)

Random forest regression 18314.442935616276


Cross Validation

In [21]:
#Cross validation on linear regression
lin_rmses = -cross_val_score(lin_reg,housing_prepared,housing_labels,scoring="neg_root_mean_squared_error", cv =10)
print(f"Linear Regression RMSE: {lin_rmse} \nLinear Regression Cross Validation:\n {pd.Series(lin_rmses).describe()}")

Linear Regression RMSE: 69050.56219504568 
Linear Regression Cross Validation:
 count       10.000000
mean     69204.322755
std       2500.382157
min      65318.224029
25%      67124.346106
50%      69404.658178
75%      70697.800632
max      73003.752739
dtype: float64


In [22]:
# Cross validation on decision tree regression
decision_tree_rmses = -cross_val_score(decision_tree_reg,housing_prepared,housing_labels, scoring="neg_root_mean_squared_error", cv=10)
print(f"Decision Tree Regression RMSE: {decision_tree_rmse} \nDecision Tree Cross validation:\n {pd.Series(decision_tree_rmses).describe()}") 

Decision Tree Regression RMSE: 0.0 
Decision Tree Cross validation:
 count       10.000000
mean     69725.599214
std       2175.672927
min      65431.828618
25%      68664.104192
50%      69482.315675
75%      70769.226679
max      73209.540772
dtype: float64


In [25]:
# Cross validation on Random forest regression
random_rmses = -cross_val_score(random_reg, housing_prepared, housing_labels, scoring="neg_root_mean_squared_error", cv=10)
print(f"Random forest Regression RMSE: {random_rmse} \nRandom Forest Cross validation: \n{pd.Series(random_rmses).describe()}")

Random forest Regression RMSE: 18314.442935616276 
Random Forest Cross validation: 
count       10.000000
mean     49374.897223
std       2201.430937
min      45729.932891
25%      47718.653785
50%      49363.239184
75%      50697.215107
max      53087.893403
dtype: float64
